# PDF ingestion and retrieval

One linear flow: PDF → GROBID TEI → parsed papers → chunks → embeddings → cross-paper retrieval and generation.

In [1]:
from pathlib import Path
import sys

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'src').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PARSED_DIR = PROJECT_ROOT / 'data' / 'parsed'
pdf_files = sorted(RAW_DIR.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDF files found in {RAW_DIR}')

print('Project root:', PROJECT_ROOT)
print('PDFs:', [path.name for path in pdf_files])

Project root: c:\Users\SWS\Desktop\projects\stage\research-paper-assistant
PDFs: ['attention_is_all_you_need.pdf', 'bert.pdf', 'rag.pdf']


## 1–4. Ingest, parse, and chunk every PDF

In [2]:
from src.ingestion.pipeline import ingest_papers

papers, chunks = ingest_papers(raw_dir=RAW_DIR, parsed_dir=PARSED_DIR)
print('Papers:', len(papers))
print('Chunks:', len(chunks))

Found 3 papers.

Processing: attention_is_all_you_need.pdf
TEI already exists.
Title: Attention Is All You Need
Chunks: 24

Processing: bert.pdf
TEI already exists.
Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Chunks: 28

Processing: rag.pdf
TEI already exists.
Title: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Chunks: 28
Papers: 3
Chunks: 80


In [3]:
for paper in papers:
    print('=' * 80)
    print('Title:', paper['title'])
    print('Authors:', ', '.join(paper['authors'][:5]))
    print('Top-level sections:', len(paper['sections']))

for chunk in chunks[:3]:
    print('=' * 80)
    print(chunk['document'], '|', chunk['section'])
    print(chunk['text'][:500])

Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Google Brain, Niki Parmar, Jakob Uszkoreit
Top-level sections: 24
Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Authors: Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova
Top-level sections: 27
Title: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Authors: Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, Vladimir Karpukhin
Top-level sections: 31
attention_is_all_you_need.pdf | Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5] . Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15] . Recurrent models typically facto

## 5–6. Embed and index the chunks

In [4]:
from src.embeddings.embedder import Embedder

embedder = Embedder()
texts = [f"{c['paper_title']}\n{c['section']}\n{c['text']}" for c in chunks]
embeddings = embedder.embed_texts(texts)
print('Embedding matrix:', embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding matrix: (80, 384)


In [5]:
from qdrant_client import QdrantClient
from src.retrieval.vector_store import VectorStore

vector_store = VectorStore(collection_name='research_papers_multi')
try:
    vector_store.client.get_collections()
    print('Using Qdrant at http://localhost:6333')
except Exception:
    vector_store.client = QdrantClient(':memory:')
    print('Qdrant unavailable; using an in-memory vector store')

vector_store.create_collection(vector_size=embeddings.shape[1])
vector_store.add_chunks(chunks, embeddings)
print('Indexed chunks:', len(chunks))

Qdrant unavailable; using an in-memory vector store
Indexed chunks: 80


c:\Users\SWS\Desktop\projects\stage\research-paper-assistant\venv\Lib\site-packages\qdrant_client\qdrant_remote.py:292: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


## 7. Retrieve and rerank

In [6]:
from src.retrieval.reranker import Reranker

reranker = Reranker()
query = 'What are the two sub-layers in each Transformer encoder layer?'
query_embedding = embedder.embed_query(query)
results = vector_store.search(query_embedding, limit=10)
reranked_results = reranker.rerank(query=query, results=results, top_k=5)

for rank, result in enumerate(reranked_results, start=1):
    payload = result['payload']
    print('=' * 80)
    print(f"{rank}. {payload['paper_title']} | {payload['section']} | score={result['score']:.4f}")
    print(payload['text'][:500])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

1. Attention Is All You Need | Encoder and Decoder Stacks | score=6.8785
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1] . That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate the
2. Attention Is All You Need | Position-wise Feed-Forward Networks | score=6.8531
In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully connected feed-forward network, which is applied to each position separately and identically. This consists of two linear transformations with a ReLU activation in between. While the linear transformations are the same across different position

## 8. Test cross-paper retrieval

Retrieve a larger candidate set and inspect whether the reranked results contain evidence from both BERT and the original Transformer paper.

In [7]:
from src.retrieval.comparison_retriever import retrieve_for_comparison

targets = [
    {
        'query': 'What is the architecture of BERT?',
        'paper_title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding',
    },
    {
        'query': 'What is the architecture of the original Transformer?',
        'paper_title': 'Attention Is All You Need',
    },
]

comparison_results = retrieve_for_comparison(
    vector_store=vector_store,
    embedder=embedder,
    reranker=reranker,
    targets=targets,
    per_query_limit=10,
    per_query_top_k=3,
)

for i, result in enumerate(comparison_results, start=1):
    print('=' * 80)
    print('Result:', i)
    print('Retrieval query:', result['retrieval_query'])
    print('Target paper:', result['target_paper'])
    print('Paper:', result['payload']['paper_title'])
    print('Section:', result['payload']['section'])
    print('Score:', result['score'])
    print()
    print(result['payload']['text'][:700])

papers_retrieved = {result['payload']['paper_title'] for result in comparison_results}
print('\nPapers represented:', len(papers_retrieved))
for title in sorted(papers_retrieved):
    print('-', title)

Result: 1
Retrieval query: What is the architecture of BERT?
Target paper: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Paper: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Section: BERT
Score: 3.5383877754211426

We introduce BERT and its detailed implementation in this section. There are two steps in our framework: pre-training and fine-tuning. During pre-training, the model is trained on unlabeled data over different pre-training tasks. For finetuning, the BERT model is first initialized with the pre-trained parameters, and all of the parameters are fine-tuned using labeled data from the downstream tasks. Each downstream task has separate fine-tuned models, even though they are initialized with the same pre-trained parameters. The question-answering example in Figure 1 will serve as a running example for this section. A distinctive feature of BERT is its unified architecture across different tasks.
Result: 2
Retr

## 9. Generate a cross-paper comparison

This uses the same retrieval and generation functions with a comparison question.

In [8]:
import importlib
import src.generation.generator as generator_module

importlib.reload(generator_module)
generate_comparison_answer = generator_module.generate_comparison_answer

question = 'Compare BERT with the original Transformer architecture.'

comparison_answer = generate_comparison_answer(
    question=question,
    comparison_results=comparison_results,
    model='llama3.2:3b',
    vector_store=vector_store,
    embedder=embedder,
    reranker=reranker,
)
print('DRAFT ANSWER:')
print(comparison_answer['draft_answer'])
print('\nSUPPORTED CLAIMS:')
for claim in comparison_answer['supported_claims']:
    print('-', claim)
print('\nPARTIALLY SUPPORTED CLAIMS:')
for claim in comparison_answer['partially_supported_claims']:
    print('-', claim)
print('\nREJECTED CLAIMS:')
for claim in comparison_answer['rejected_claims']:
    print('-', claim)
print('\nVERIFICATION ERRORS:')
for claim in comparison_answer['verification_errors']:
    print('-', claim)
print('\nFINAL NORMALIZED FACT SET:')
for fact in comparison_answer['final_fact_set']:
    print('-', fact)
print('\nFINAL GROUNDED ANSWER:')
print(comparison_answer['final_answer'])

Extracted claims: 9
Unique claims: 9
Cache hits: 0
Verified claims: 9
Supported: 6
Partially supported: 0
Unsupported: 2
Errors: 1
Evidence-routing failures: 0
Malformed partial verdicts: 0
Final facts: 6

Original draft section: BERT
Original bullet: BERT is a deep bidirectional Transformer encoder based on the original Transformer architecture (Vaswani et al., 2017).
Atomic claim: BERT is a deep bidirectional Transformer encoder based on the original Transformer architecture
Primary entity: BERT
Referenced entities: Transformer
Relation type: REFERENCE_ONLY
Resolved paper: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Evidence papers: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Evidence used:
Source 1 - BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding - BERT
Source 2 - BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding - Effect of Pre-training Tasks
So

## 10. Evaluate retrieval, grounding, and comparison answers

The reusable evaluation code lives in `evaluation/`; this notebook only orchestrates it. The retrieval cell compares vector search with cross-encoder reranking. The generation cell runs the complete Ollama claim-verification pipeline for all evaluation questions and can take several minutes.

In [9]:
from collections import Counter

from evaluation.evaluation_utils import load_questions

EVALUATION_DIR = PROJECT_ROOT / 'evaluation'
RESULTS_DIR = EVALUATION_DIR / 'results'
evaluation_questions = load_questions(EVALUATION_DIR / 'questions.json')

print('Evaluation questions:', len(evaluation_questions))
for question_type, count in sorted(Counter(q['type'] for q in evaluation_questions).items()):
    print(f'- {question_type}: {count}')

Evaluation questions: 18
- comparison: 3
- fact: 13
- insufficient_evidence: 2


In [10]:
from evaluation.evaluate_retrieval import evaluate_retrieval_dataset

retrieval_evaluation = evaluate_retrieval_dataset(
    questions=evaluation_questions,
    vector_store=vector_store,
    embedder=embedder,
    reranker=reranker,
)

retrieval_summary = retrieval_evaluation['summary']
print('=' * 60)
print('RETRIEVAL EVALUATION')
print('=' * 60)
print('Standard questions:', retrieval_summary['questions'])
print('Comparison questions:', retrieval_summary['comparison_questions'])
print('Before reranking:', retrieval_summary['before_reranking'])
print('After reranking:', retrieval_summary['after_reranking'])
print('Reranking improvement:', retrieval_summary['reranking_improvement'])
print('Comparison paper coverage:', retrieval_summary['comparison_paper_coverage'])

retrieval_failures = [
    item for item in retrieval_evaluation['per_question']
    if item.get('likely_failure_stage')
]
print('\nRetrieval failures:', len(retrieval_failures))
for item in retrieval_failures:
    print(f"- {item['id']} | {item['likely_failure_stage']} | {item['question']}")

RETRIEVAL EVALUATION
Standard questions: 13
Comparison questions: 3
Before reranking: {'hit@1': 0.5384615384615384, 'hit@3': 1, 'hit@5': 1, 'mrr': 0.7307692307692307}
After reranking: {'hit@1': 0.9230769230769231, 'hit@3': 0.9230769230769231, 'hit@5': 1, 'mrr': 0.9384615384615385}
Reranking improvement: {'hit@1': 0.3846153846153847, 'hit@3': -0.07692307692307687, 'mrr': 0.20769230769230773}
Comparison paper coverage: 1.0

Retrieval failures: 0


In [11]:
from evaluation.evaluate_generation import evaluate_generation_dataset

generation_evaluation = evaluate_generation_dataset(
    questions=evaluation_questions,
    vector_store=vector_store,
    embedder=embedder,
    reranker=reranker,
    model='llama3.2:3b',
)

generation_summary = generation_evaluation['summary']
print('=' * 60)
print('GENERATION EVALUATION')
print('=' * 60)
for metric, value in generation_summary.items():
    print(f'{metric}: {value}')

generation_failures = [
    item for item in generation_evaluation['per_question']
    if item.get('likely_failure_stage')
]
print('\nGeneration failures:', len(generation_failures))
for item in generation_failures:
    print('\n' + item['id'], '-', item['likely_failure_stage'])
    print('Question:', item['question'])
    if item.get('missing_expected_facts'):
        print('Missing expected facts:')
        for fact in item['missing_expected_facts']:
            print('-', fact)

KeyboardInterrupt: 

In [ ]:
from evaluation.evaluation_utils import save_json, utc_timestamp
from evaluation.run_evaluation import print_summary

evaluation_summary = {
    'retrieval': retrieval_evaluation['summary'],
    'generation': generation_evaluation['summary'],
}
timestamp = utc_timestamp()
for name, value in (
    ('retrieval_results', retrieval_evaluation),
    ('generation_results', generation_evaluation),
    ('summary', evaluation_summary),
):
    save_json(RESULTS_DIR / f'{name}.json', value)
    save_json(RESULTS_DIR / f'{name}_{timestamp}.json', value)

print_summary(retrieval_evaluation, generation_evaluation)
print('\nSaved evaluation results to:', RESULTS_DIR)